# Week 3 (Supplemental): Claude Code — Local Dev Power Tools

**Lab companion to [week_03_local_dev.md](../agenda/week_03_local_dev.md)**

In this lab you will:
1. Inspect and extend the real `.claude/` configuration already in this repo
2. Read and test the live hook scripts
3. Analyze the skill and agent definitions — then write a new one
4. Examine the MCP server configuration
5. Run a full audit of the complete setup

> **Note:** Unlike most labs, the files you're working with already exist and are active.
> Exercises ask you to read, test, extend, and reflect — not build from scratch.

In [1]:
# Setup
import os, json, stat, subprocess, re
from pathlib import Path

REPO_ROOT = Path(os.getcwd()).parent
CLAUDE_DIR = REPO_ROOT / '.claude'

assert CLAUDE_DIR.exists(), f'.claude/ not found at {CLAUDE_DIR}'

def tree(path: Path, indent: int = 0) -> None:
    prefix = '  ' * indent
    for item in sorted(path.iterdir()):
        if item.name.startswith('.'):
            continue
        print(f'{prefix}{item.name}{'/' if item.is_dir() else ''}')
        if item.is_dir():
            tree(item, indent + 1)

print(f'Repo root: {REPO_ROOT}')
print(f'\n.claude/ contents:')
tree(CLAUDE_DIR)
print(f'\n.mcp.json exists: {(REPO_ROOT / ".mcp.json").exists()}')
print('✓ Ready!')

Repo root: /Users/volod34/PycharmProjects/ai_track

.claude/ contents:
agents/
  code-reviewer.md
  doc-researcher.md
  notebook-validator.md
  python-linter.md
commands/
  eval.md
  orchestrate.md
  plan.md
  skill-create.md
  tdd.md
contexts/
  architecture.md
  lab-conventions.md
  models.md
hooks/
  audit-log.sh
  detect-secrets.sh
  format-python.sh
  load-env.sh
  run-tests.sh
  validate-notebook.sh
rules/
  api-design.md
  notebooks.md
  python.md
  security.md
  testing.md
settings.json
settings.local.json
skills/
  commit/
    SKILL.md
  debug/
    SKILL.md
  explain/
    SKILL.md
  lab-status/
    SKILL.md
  new-week/
    SKILL.md
  review/
    SKILL.md

.mcp.json exists: True
✓ Ready!


---
## Part 1: Memory — CLAUDE.md, Contexts, and Rules

This repo uses **three layers** of memory configuration:
- `CLAUDE.md` — top-level, imports contexts with `@`
- `.claude/contexts/` — detailed reference material imported into every session
- `.claude/rules/` — path-scoped instructions loaded only for matching files

### Exercise 1.1 — Inspect the Memory Stack

Map exactly what Claude loads at session start when working in this repo.

In [ ]:
# Read CLAUDE.md and find all @-imports
claude_md = (REPO_ROOT / 'CLAUDE.md').read_text()
imports = re.findall(r'^@(.+)$', claude_md, re.MULTILINE)

print('CLAUDE.md @-imports:')
total_tokens = 0
for imp in imports:
    path = REPO_ROOT / imp
    if path.exists():
        content = path.read_text()
        # Strip HTML comments (not loaded into context)
        stripped = re.sub(r'<!--.*?-->', '', content, flags=re.DOTALL)
        lines = len(stripped.splitlines())
        # Rough token estimate: ~4 chars per token
        tokens = len(stripped) // 4
        total_tokens += tokens
        print(f'  {imp} ({lines} lines, ~{tokens} tokens)')
    else:
        print(f'  {imp} — MISSING')

claude_tokens = len(claude_md) // 4
total_tokens += claude_tokens
print(f'\nCLAUDE.md itself: ~{claude_tokens} tokens')
print(f'Total session start context: ~{total_tokens} tokens')

print('\n--- Rules (loaded lazily by path) ---')
for rule in sorted((CLAUDE_DIR / 'rules').glob('*.md')):
    content = rule.read_text()
    fm = re.search(r'^---\n(.*?)\n---', content, re.DOTALL)
    paths_val = 'always loaded'
    if fm and 'paths:' in fm.group(1):
        paths = re.findall(r'- "(.+?)"', fm.group(1))
        paths_val = ', '.join(paths)
    print(f'  {rule.name:25s} → {paths_val}')

### Exercise 1.2 — Read the Context Files

Read each context file and identify the key facts Claude gets from each one.

In [ ]:
contexts_dir = CLAUDE_DIR / 'contexts'
for ctx_file in sorted(contexts_dir.glob('*.md')):
    content = ctx_file.read_text()
    lines = content.splitlines()
    # Extract H2 and H3 headers as the structure
    headers = [l for l in lines if l.startswith('##')]
    print(f'=== {ctx_file.name} ({len(lines)} lines) ===')
    for h in headers:
        print(f'  {h}')
    print()

In [ ]:
# TODO: Answer these questions by reading the context files above.
# Fill in the correct values:

answers = {
    'default_model_for_labs': None,          # What model do labs use by default?
    'local_model_provider': None,            # What env var switches to local models?
    'portfolio_1_weeks': None,               # Which weeks build Portfolio #1?
    'env_loading_pattern': None,             # How do notebooks load .env?
    'details_block_rule': None,              # What must never happen to <details> blocks?
}

# Check answers
expected = {
    'default_model_for_labs': 'gpt-4o-mini',
    'local_model_provider': 'AI_PROVIDER=ollama',
    'portfolio_1_weeks': 'Weeks 5-8',
    'env_loading_pattern': 'load_dotenv(os.path.join(os.path.dirname(os.getcwd()), ".env"))',
    'details_block_rule': 'Never remove or reorganize <details> hint/solution blocks',
}

all_pass = True
for k, v in answers.items():
    if v is None:
        print(f'  TODO: {k}')
        all_pass = False
    elif expected[k].lower() in str(v).lower():
        print(f'  PASS: {k}')
    else:
        print(f'  FAIL: {k} — got "{v}"')
        all_pass = False

if all_pass:
    print('\nAll correct!')

<details>
<summary>✅ <b>Solution</b> (click to expand)</summary>

```python
answers = {
    'default_model_for_labs': 'gpt-4o-mini',
    'local_model_provider': 'AI_PROVIDER=ollama',
    'portfolio_1_weeks': 'Weeks 5-8',
    'env_loading_pattern': 'load_dotenv(os.path.join(os.path.dirname(os.getcwd()), ".env"))',
    'details_block_rule': 'Never remove or reorganize <details> hint/solution blocks',
}
```

</details>

### Exercise 1.3 — Extend a Context File

The `models.md` context file is missing Ollama embedding models for weeks 3-6. Add them.

In [ ]:
# Read the current models.md
models_ctx = (CLAUDE_DIR / 'contexts' / 'models.md').read_text()
print(models_ctx)

In [ ]:
# TODO: Add a new section to models.md for embedding models.
# The section should cover:
#   - OpenAI: text-embedding-3-small (1536 dims) and text-embedding-3-large (3072 dims)
#   - Ollama: nomic-embed-text and bge-m3 (multilingual)
#   - When to use each (speed vs quality, local vs cloud)
#
# IMPORTANT: Read the existing file before editing — never overwrite existing content.
# Use string concatenation to append the new section.

new_section = """
## Embedding Models (Weeks 3-6)

# TODO: Add embedding model table and usage guidance
"""

# Append to the file (only if section not already present)
models_path = CLAUDE_DIR / 'contexts' / 'models.md'
current = models_path.read_text()
if 'Embedding Models' not in current:
    models_path.write_text(current + new_section)
    print('Section added.')
else:
    print('Embedding section already present.')
print(models_path.read_text()[-800:])

<details>
<summary>✅ <b>Solution</b> (click to expand)</summary>

```python
new_section = """
## Embedding Models (Weeks 3-6)

| Model | Provider | Dimensions | Use case |
|-------|----------|-----------|----------|
| `text-embedding-3-small` | OpenAI | 1536 | Default — fast, cost-effective |
| `text-embedding-3-large` | OpenAI | 3072 | Higher quality for production RAG |
| `nomic-embed-text` | Ollama | 768 | Local, English-only |
| `bge-m3` | Ollama | 1024 | Local, multilingual (100+ languages) |

**Dimension reduction:** `text-embedding-3` models support `dimensions=N` to reduce storage:
```python
client.embeddings.create(model="text-embedding-3-small", input=text, dimensions=256)
```
Use 256-512 dims for fast similarity search when recall is less critical.
"""
```

</details>

---
## Part 2: Hooks — Reading and Testing the Real Scripts

`.claude/hooks/` contains six executable scripts, all wired into `.claude/settings.json`.
These are live — they run every time Claude edits a file or stops responding.

### Exercise 2.1 — Read the Settings and Map the Hook Pipeline

In [ ]:
# Read settings.json and map each event → scripts
settings = json.loads((CLAUDE_DIR / 'settings.json').read_text())

print('Hook pipeline:')
for event, groups in settings.get('hooks', {}).items():
    for group in groups:
        matcher = group.get('matcher', '(all tools)')
        print(f'\n  [{event}] matcher: {matcher}')
        for hook in group.get('hooks', []):
            cmd = hook.get('command', hook.get('url', '?'))
            async_ = ' (async)' if hook.get('async') else ' (blocking)'
            # Shorten path for display
            cmd_short = cmd.replace('"$CLAUDE_PROJECT_DIR"/.claude/hooks/', '')
            print(f'    → {cmd_short}{async_}')

In [ ]:
# Verify all hook scripts are executable
hooks_dir = CLAUDE_DIR / 'hooks'
print('Hook scripts:')
for script in sorted(hooks_dir.glob('*.sh')):
    is_exec = os.access(script, os.X_OK)
    size = script.stat().st_size
    lines = len(script.read_text().splitlines())
    status = '✓' if is_exec else '✗ NOT EXECUTABLE'
    print(f'  {status}  {script.name:30s} {lines} lines, {size} bytes')

### Exercise 2.2 — Test the Secret Detection Hook

The hook at `.claude/hooks/detect-secrets.sh` is live. Test it by simulating hook inputs.

In [ ]:
def run_hook(script_name: str, stdin_data: dict) -> dict:
    """Run a hook script with simulated JSON input."""
    script = CLAUDE_DIR / 'hooks' / script_name
    result = subprocess.run(
        [str(script)],
        input=json.dumps(stdin_data),
        capture_output=True, text=True
    )
    return {
        'exit_code': result.returncode,
        'stdout': result.stdout.strip(),
        'stderr': result.stderr.strip(),
        'blocked': result.returncode == 2
    }

test_cases = [
    ('Hardcoded password — SHOULD BLOCK',
     {'tool_input': {'new_string': 'password = "my_secret_123"'}},
     True),
    ('Env var reference — SHOULD ALLOW',
     {'tool_input': {'new_string': 'password = os.environ["DB_PASSWORD"]'}},
     False),
    ('Hardcoded API key — SHOULD BLOCK',
     {'tool_input': {'new_string': 'api_key = "sk-abc123xyz"'}},
     True),
    ('os.getenv call — SHOULD ALLOW',
     {'tool_input': {'new_string': 'token = os.getenv("GITHUB_TOKEN", "")'}},
     False),
    ('Normal code — SHOULD ALLOW',
     {'tool_input': {'new_string': 'def compute_similarity(a: list, b: list) -> float:'}},
     False),
]

all_pass = True
for name, payload, expect_blocked in test_cases:
    result = run_hook('detect-secrets.sh', payload)
    passed = result['blocked'] == expect_blocked
    all_pass = all_pass and passed
    got = 'BLOCKED' if result['blocked'] else 'ALLOWED'
    exp = 'BLOCKED' if expect_blocked else 'ALLOWED'
    status = 'PASS' if passed else 'FAIL'
    print(f'[{status}] {name}')
    print(f'       Expected: {exp} | Got: {got} (exit {result["exit_code"]})')
    if result['stderr']:
        print(f'       Reason: {result["stderr"]}')

print(f'\n{"All tests passed!" if all_pass else "Some tests FAILED — read detect-secrets.sh"}')

In [ ]:
# Read the actual script to understand how it works
print((CLAUDE_DIR / 'hooks' / 'detect-secrets.sh').read_text())

### Exercise 2.3 — Extend detect-secrets.sh

The current script misses one common pattern: `SECRET_KEY = "..."` (uppercase variable names). Add support for it.

In [ ]:
# First, verify the gap: does the current hook catch uppercase?
gap_test = {
    'tool_input': {'new_string': 'SECRET_KEY = "django-insecure-abc123"'}
}
result = run_hook('detect-secrets.sh', gap_test)
print(f'Uppercase SECRET_KEY: {"BLOCKED" if result["blocked"] else "ALLOWED (GAP!)"}' )
print(f'The hook should BLOCK this but currently {"does" if result["blocked"] else "does NOT"}')

In [ ]:
# TODO: Extend detect-secrets.sh to also catch:
# 1. Uppercase variable names: SECRET_KEY="...", DATABASE_URL="...", AUTH_TOKEN="..."
# 2. Django-style: SECRET_KEY = "django-insecure-..."
#
# Strategy: add a second pattern check after the first one.
# The existing SAFE_PATTERN (env var references) should still whitelist them.
#
# Read the current script, modify the SECRET_PATTERN variable, write it back.

script_path = CLAUDE_DIR / 'hooks' / 'detect-secrets.sh'
original = script_path.read_text()

# TODO: modify `original` to improve the pattern, then write it back
# Hint: the current SECRET_PATTERN line looks like:
# SECRET_PATTERN='(password|api_key|...)\s*=\s*...'
# Add uppercase patterns: [A-Z_]+_(KEY|SECRET|TOKEN|PASSWORD)

# modified = original.replace(...)
# script_path.write_text(modified)
# script_path.chmod(script_path.stat().st_mode | 0o111)

print('Current SECRET_PATTERN line:')
for line in original.splitlines():
    if 'SECRET_PATTERN' in line:
        print(f'  {line}')

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

Add a second `grep` check for uppercase patterns. The safest approach is to add a second `UPPERCASE_PATTERN` after the existing check:

```bash
# After the existing SECRET_PATTERN block, add:
UPPERCASE_PATTERN='[A-Z][A-Z0-9_]*(KEY|SECRET|TOKEN|PASSWORD|CREDENTIALS)\s*=\s*["\x27][^"\x27${}][^"\x27]{2,}["\x27]'

if echo "$CONTENT" | grep -iqE "$UPPERCASE_PATTERN"; then
    # same whitelist check...
fi
```

</details>

<details>
<summary>✅ <b>Solution</b> (click to expand)</summary>

```python
# Replace the SECRET_PATTERN line with a combined pattern
old_pattern = "SECRET_PATTERN='(password|api_key|api_secret|secret_key|token|credential|auth_key)\\s*=\\s*"
new_pattern = "SECRET_PATTERN='(password|api_key|api_secret|secret_key|token|credential|auth_key|[A-Z][A-Z0-9_]*(KEY|SECRET|TOKEN|PASSWORD|CREDENTIALS))\\s*=\\s*"

modified = original.replace(old_pattern, new_pattern)
script_path.write_text(modified)
script_path.chmod(script_path.stat().st_mode | 0o111)

# Verify
result = run_hook('detect-secrets.sh', {'tool_input': {'new_string': 'SECRET_KEY = "django-insecure-abc"'}})
print(f'After fix: {"BLOCKED" if result["blocked"] else "still not blocked"}')
```

</details>

### Exercise 2.4 — Read the audit log

The audit-log.sh hook appends to `~/.claude/audit.jsonl` on every tool use. If you've used Claude Code in this project, there should be entries.

In [ ]:
# Simulate the audit-log hook to create a test entry
test_event = {
    'hook_event_name': 'PostToolUse',
    'session_id': 'lab-exercise-001',
    'cwd': str(REPO_ROOT),
    'tool_name': 'Edit',
    'tool_input': {'path': 'labs/week_03_local_dev.ipynb'},
}
run_hook('audit-log.sh', test_event)

# Read the audit log
audit_file = Path.home() / '.claude' / 'audit.jsonl'
if audit_file.exists():
    lines = audit_file.read_text().strip().splitlines()
    print(f'Audit log: {len(lines)} total entries')
    print('\nLast 5 entries:')
    for line in lines[-5:]:
        try:
            entry = json.loads(line)
            print(f'  {entry.get("ts", "?")} | {entry.get("tool", "?")} | {entry.get("path", entry.get("cwd", "?"))}')
        except json.JSONDecodeError:
            print(f'  [malformed] {line[:60]}')

    # Summarize by tool type
    from collections import Counter
    tools = Counter()
    for line in lines:
        try:
            tools[json.loads(line).get('tool', 'unknown')] += 1
        except:
            pass
    print('\nTool usage distribution:')
    for tool, count in tools.most_common(10):
        print(f'  {tool:20s} {count}')
else:
    print('No audit log yet.')

---
## Part 3: Skills — Reading and Extending the Slash Commands

`.claude/skills/` contains six skills that are live in Claude Code. Read them, understand their design decisions, then write a new one.

### Exercise 3.1 — Inventory All Skills

In [ ]:
def parse_skill(skill_path: Path) -> dict:
    """Parse a SKILL.md file into its components."""
    content = skill_path.read_text()
    fm_match = re.match(r'^---\n(.*?)\n---', content, re.DOTALL)
    fm = {}
    if fm_match:
        for line in fm_match.group(1).splitlines():
            if ':' in line and not line.startswith(' '):
                k, _, v = line.partition(':')
                fm[k.strip()] = v.strip()
        body = content[fm_match.end():].strip()
    else:
        body = content
    return {'frontmatter': fm, 'body': body, 'body_lines': len(body.splitlines())}

skills_dir = CLAUDE_DIR / 'skills'
print(f'{'Skill':<12} {'Invoke':<8} {'Model':<9} {'Context':<8} {'Body':>5}  Description')
print('-' * 90)
for skill_dir in sorted(skills_dir.iterdir()):
    skill_file = skill_dir / 'SKILL.md'
    if not skill_file.exists():
        continue
    parsed = parse_skill(skill_file)
    fm = parsed['frontmatter']
    name = fm.get('name', skill_dir.name)
    manual_only = 'manual' if fm.get('disable-model-invocation') == 'true' else 'auto'
    model = fm.get('model', 'inherit')
    context = fm.get('context', 'inline')
    desc = fm.get('description', '')[:50]
    print(f'/{name:<11} {manual_only:<8} {model:<9} {context:<8} {parsed["body_lines"]:>5}  {desc}')

### Exercise 3.2 — Read the `/review` Skill

Understand exactly what Claude does when you type `/review`.

In [ ]:
print((CLAUDE_DIR / 'skills' / 'review' / 'SKILL.md').read_text())

In [ ]:
# TODO: Answer these questions about the /review skill design:

review_analysis = {
    # Why is disable-model-invocation: true on /review?
    'why_manual_only': None,

    # What does allowed-tools restrict vs. what Claude can normally do?
    'what_tools_restricted': None,

    # What does $ARGUMENTS control in the skill?
    'arguments_purpose': None,

    # What are the three output sections?
    'output_sections': None,
}

for k, v in review_analysis.items():
    status = '✓' if v else 'TODO'
    print(f'[{status}] {k}: {v or "(answer here)"}')

<details>
<summary>✅ <b>Solution</b> (click to expand)</summary>

```python
review_analysis = {
    'why_manual_only':
        'Code reviews have a specific trigger (before commit/merge). Claude should not \'helpfully\' '
        'run a review every time you mention changing code.',

    'what_tools_restricted':
        'allowed-tools: Read, Bash(git *), Grep, Glob — Claude cannot Edit, Write, or run '
        'arbitrary Bash. The review is read-only and git-aware.',

    'arguments_purpose':
        '$ARGUMENTS narrows the diff to a specific file or directory. '
        'Without it, git diff HEAD is used (all recent changes).',

    'output_sections':
        'Critical (block merge), Warnings (tech debt), Suggestions (optional) — each with file:line references.',
}
```

</details>

### Exercise 3.3 — Write a `/weekly-summary` Skill

Create a new skill that summarizes git activity for the current week — commits made, files changed, tests added.

In [ ]:
# TODO: Create .claude/skills/weekly-summary/SKILL.md
#
# Requirements:
#   - name: weekly-summary
#   - Manual only (disable-model-invocation: true)
#   - Allowed tools: Bash(git *) only — no file editing
#   - Use !`command` to inject live git data:
#       - Commits since Monday: git log --oneline --since='monday'
#       - Files changed: git diff HEAD~7 --stat
#       - New test files: git log --name-only --since='monday' | grep 'test_'
#   - Output: a weekly progress report with commit count, key changes, areas covered
#   - Target: useful for a student tracking their learning progress

weekly_summary = '''---
name: weekly-summary
# TODO: fill in the rest of the frontmatter
---

# TODO: fill in the skill body
'''

skill_dir = CLAUDE_DIR / 'skills' / 'weekly-summary'
skill_dir.mkdir(exist_ok=True)
skill_path = skill_dir / 'SKILL.md'
skill_path.write_text(weekly_summary)
print(f'Created: {skill_path}')
print(skill_path.read_text())

<details>
<summary>✅ <b>Solution</b> (click to expand)</summary>

````markdown
---
name: weekly-summary
description: Summarize git activity and learning progress for the current week. Use at the end of a study session to review what was accomplished.
disable-model-invocation: true
allowed-tools: Bash(git *)
---

## This Week's Activity

**Commits since Monday:**
```
!`git log --oneline --since='monday' 2>/dev/null || echo 'No commits this week'`
```

**Files changed:**
```
!`git diff --stat HEAD~7 2>/dev/null | tail -5 || git diff --stat HEAD~1 2>/dev/null | tail -5`
```

**New/modified lab notebooks:**
```
!`git log --name-only --since='monday' --diff-filter=AM --pretty='' 2>/dev/null | grep '\.ipynb' | sort -u`
```

## Your Task

Write a concise weekly progress report covering:

1. **Commit count** and what was worked on (infer from commit messages)
2. **Labs touched** — which weeks' content was explored
3. **Key concepts practiced** — based on filenames and commit messages
4. **Momentum assessment** — steady / needs more reps / strong week
5. **Suggested focus** for next session — what to tackle next based on the curriculum

Keep it to one short paragraph per section. Tone: encouraging coach, not bureaucratic report.
````

</details>

---
## Part 4: Agents — Reading and Extending the Subagents

### Exercise 4.1 — Inventory All Agents

In [ ]:
def parse_agent(agent_path: Path) -> dict:
    """Parse an agent .md file."""
    content = agent_path.read_text()
    fm_match = re.match(r'^---\n(.*?)\n---', content, re.DOTALL)
    fm = {}
    if fm_match:
        for line in fm_match.group(1).splitlines():
            if ':' in line and not line.startswith(' '):
                k, _, v = line.partition(':')
                fm[k.strip()] = v.strip()
    body = content[fm_match.end():].strip() if fm_match else content
    return {'fm': fm, 'body_lines': len(body.splitlines()), 'body': body[:200]}

agents_dir = CLAUDE_DIR / 'agents'
print(f'{'Agent':<22} {'Model':<9} {'Memory':<9} {'Tools':<35} {'Body'}')
print('-' * 90)
for f in sorted(agents_dir.glob('*.md')):
    a = parse_agent(f)
    fm = a['fm']
    tools = fm.get('tools', '(inherit all)')
    denied = fm.get('disallowedTools', '')
    tools_display = f'deny:{denied}' if denied else tools[:33]
    print(f'{f.stem:<22} {fm.get("model","inherit"):<9} {fm.get("memory","-"):<9} {tools_display:<35} {a["body_lines"]}L')

### Exercise 4.2 — Compare Tool Restriction Strategies

In [ ]:
# Read both the code-reviewer (allowlist) and python-linter (denylist) agents
# and understand why each strategy was chosen.

reviewer = parse_agent(agents_dir / 'code-reviewer.md')
linter = parse_agent(agents_dir / 'python-linter.md')

print('=== code-reviewer.md — ALLOWLIST strategy ===')
print(f'tools: {reviewer["fm"].get("tools")}')
print(f'disallowedTools: {reviewer["fm"].get("disallowedTools", "(not set)")}')
print()
print('=== python-linter.md — DENYLIST strategy ===')
print(f'tools: {linter["fm"].get("tools", "(not set)")}')
print(f'disallowedTools: {linter["fm"].get("disallowedTools")}')
print()

# TODO: Explain the tradeoff between allowlist and denylist in one sentence each:
allowlist_tradeoff = None  # When is allowlist better?
denylist_tradeoff = None   # When is denylist better?

print(f'Allowlist tradeoff: {allowlist_tradeoff or "(your answer)"}')
print(f'Denylist tradeoff:  {denylist_tradeoff or "(your answer)"}')

<details>
<summary>✅ <b>Solution</b> (click to expand)</summary>

```python
allowlist_tradeoff = (
    "Use allowlist (tools:) when you know exactly what the agent needs — "
    "it's safer because new tools added to Claude won't automatically be available."
)
denylist_tradeoff = (
    "Use denylist (disallowedTools:) when the agent needs most tools but must be "
    "blocked from a specific dangerous few — simpler to maintain as Claude gains new tools."
)
```

The code-reviewer uses an **allowlist** because it knows exactly what it needs: read, search, git. Allowing anything else (like Bash arbitrary commands or Write) would make it a reviewer that can silently modify code — defeating the purpose.

The python-linter uses a **denylist** because it needs Bash (to run ruff/mypy), Read (to inspect files), and Glob — essentially everything except the write tools. Listing all those individually would be tedious.

</details>

### Exercise 4.3 — Write a `/dependency-auditor` Agent

Create a new subagent that checks for outdated or vulnerable dependencies.

In [ ]:
# TODO: Create .claude/agents/dependency-auditor.md
#
# Design decisions to make:
#   - What model? (it's mostly running commands and reading output)
#   - allowlist or denylist? (it needs Bash and Read, but NOT Edit or Write)
#   - memory? (project — it can remember which deps had issues before)
#   - What commands does it run?
#       uv run pip list --outdated (check outdated)
#       uv tree (see dependency graph)
#       cat pyproject.toml (read declared deps)
#   - Output: table of outdated packages, any known CVEs if pip-audit available

dep_auditor = '''---
name: dependency-auditor
# TODO: fill in frontmatter
---

# TODO: fill in system prompt
'''

agent_path = agents_dir / 'dependency-auditor.md'
agent_path.write_text(dep_auditor)
print(f'Created: {agent_path}')

<details>
<summary>✅ <b>Solution</b> (click to expand)</summary>

```markdown
---
name: dependency-auditor
description: Check Python dependencies for outdated versions and security vulnerabilities. Use before releases, after merging dependency updates, or when security is a concern.
tools: Bash(uv *), Bash(cat *), Read
model: haiku
memory: project
---

You are a dependency security auditor. Run commands, report findings — never modify files.

## Audit steps

```bash
# 1. Current declared dependencies
cat pyproject.toml

# 2. Outdated packages
uv run pip list --outdated 2>&1

# 3. Dependency tree
uv tree 2>&1 | head -50

# 4. Security scan (if available)
uv run pip-audit 2>&1 || echo 'pip-audit not installed — run: uv add pip-audit'
```

## Output format

### Outdated Packages
| Package | Current | Latest | Risk |
|---------|---------|--------|------|
| openai | 1.3.0 | 1.8.0 | low (minor) |

### Security Findings
- CVE or "none found"

### Recommendation
- `safe to update` / `update with care` / `urgent: security patch`
- Specific `uv add package==version` commands to run

## Memory maintenance
Save packages that have previously caused breakage when updated.
```

</details>

---
## Part 5: MCP — Model Context Protocol Servers

### Exercise 5.1 — Read and Understand `.mcp.json`

In [ ]:
# Read and display the MCP config
mcp_config = json.loads((REPO_ROOT / '.mcp.json').read_text())

print('MCP Servers configured:')
for name, cfg in mcp_config['mcpServers'].items():
    cmd = ' '.join([cfg.get('command', '')] + cfg.get('args', []))
    env_vars = list(cfg.get('env', {}).keys())
    desc = cfg.get('description', '')
    print(f'\n  [{name}]')
    print(f'    type: {cfg.get("type")}')
    print(f'    command: {cmd[:70]}')
    if env_vars:
        print(f'    env required: {env_vars}')
    print(f'    purpose: {desc}')

In [ ]:
# TODO: For each MCP server, identify the use case in THIS curriculum.
# Think: when in weeks 1-16 would a student want to use each server?

mcp_use_cases = {
    'filesystem': None,   # When would you use MCP filesystem vs. Claude's built-in Read/Write?
    'github': None,       # Which week or project uses GitHub integration?
    'memory': None,       # What's different from Claude's auto-memory?
    'brave-search': None, # When would a student need web search?
    'fetch': None,        # vs. WebFetch built-in — what's different?
}

for server, use_case in mcp_use_cases.items():
    status = '✓' if use_case else 'TODO'
    print(f'[{status}] {server}: {use_case or "(your answer)"}')

<details>
<summary>✅ <b>Solution</b> (click to expand)</summary>

```python
mcp_use_cases = {
    'filesystem': (
        'MCP filesystem exposes structured tools (read_file, write_file, list_dir) that '
        'other MCP clients can use. Claude\'s built-in tools are equivalent for interactive '
        'use, but MCP enables programmatic access from other agents or systems.'
    ),
    'github': (
        'Weeks 12-16 (AI Agent, Capstone) — when the agent needs to read issue descriptions, '
        'create PRs, or list repository contents as part of an automated workflow.'
    ),
    'memory': (
        'Structured key-value store — different from auto-memory (which is free-form markdown). '
        'Useful when agents need to store and retrieve specific structured facts reliably, '
        'like caching embedding results or storing evaluation scores.'
    ),
    'brave-search': (
        'Weeks 5-8 when building RAG — researching what data sources to include. '
        'Week 11 (LangGraph) when the agent needs to look up current information. '
        'Any time Claude needs facts beyond its training cutoff.'
    ),
    'fetch': (
        'Similar to WebFetch but as an MCP tool — can be scoped to specific subagents. '
        'Useful for a web-scraping subagent that should only fetch, not search.'
    ),
}
```

</details>

### Exercise 5.2 — Design a Course-specific MCP Server

MCP servers expose custom tools. Design (but don't implement) an MCP server that would be genuinely useful for students working through this curriculum.

In [ ]:
# TODO: Design a custom MCP server for the ai_track curriculum.
# Fill in the design:

custom_mcp_design = {
    'name': None,           # What would you call it?
    'tools': [],            # List 3-5 tool names it would expose
    'tool_descriptions': {},# What does each tool do?
    'why_mcp_not_skill': None, # Why is this better as an MCP server than a skill?
}

# Example structure:
# custom_mcp_design = {
#     'name': 'curriculum-tracker',
#     'tools': ['get_week_objectives', 'mark_exercise_complete', 'get_next_exercise'],
#     'tool_descriptions': {
#         'get_week_objectives': 'Returns learning objectives for a given week number',
#         ...
#     },
#     'why_mcp_not_skill': 'Skills are prompt-based; this needs structured data access and state persistence'
# }

print('Your custom MCP server design:')
for k, v in custom_mcp_design.items():
    print(f'  {k}: {v or "(fill this in)"}')

---
## Part 5.5: Commands — Structured Workflow Macros

`.claude/commands/` contains five command files that become `/name` slash commands. Unlike skills, commands are simple `.md` files — no YAML frontmatter, no tool restrictions. Their power comes from **dynamic context injection** with `` !`shell command` `` syntax.

This section: read the five commands, understand what makes them different from skills, and write a new command.

### Exercise 5.5.1 — Inventory the Commands Directory

Map all five commands, their invocation syntax, and their core workflow enforcement.

In [ ]:
commands_dir = CLAUDE_DIR / 'commands'

def parse_command(cmd_path: Path) -> dict:
    """Extract key info from a command .md file."""
    content = cmd_path.read_text()
    lines = content.strip().splitlines()
    # Title line: # /name — description
    title = lines[0].lstrip('# ').strip() if lines else ''
    # Count !`...` injections
    injections = [l.strip() for l in lines if l.strip().startswith('!`')]
    # Has explicit STOP instruction?
    has_stop = 'STOP' in content.upper() and 'WAIT' in content.upper()
    return {
        'name': cmd_path.stem,
        'title': title,
        'injections': injections,
        'has_stop_gate': has_stop,
        'lines': len(lines),
    }

print(f'Commands directory: {commands_dir}')
print(f'Files: {sorted(f.name for f in commands_dir.glob("*.md"))}\n')

for cmd_file in sorted(commands_dir.glob('*.md')):
    info = parse_command(cmd_file)
    print(f'/{info["name"]}')
    print(f'  Title:         {info["title"]}')
    print(f'  Lines:         {info["lines"]}')
    print(f'  !` injections: {len(info["injections"])}')
    print(f'  Has stop gate: {info["has_stop_gate"]}')
    print()

### Exercise 5.5.2 — Read `/plan` and Understand the Stop Gate

The `/plan` command is the most architecturally important — it prevents the common failure mode where Claude generates a plan and immediately starts implementing it. Read the command and answer the questions below.

In [ ]:
print((CLAUDE_DIR / 'commands' / 'plan.md').read_text())

In [ ]:
# TODO: Answer these questions about /plan's design:

plan_analysis = {
    # Why does the plan command end with "STOP HERE. Wait for confirmation."?
    # What failure mode does this prevent?
    'why_stop_gate': '...',

    # What two things does !`git log --oneline -10` inject into Claude's context?
    # (think: what information does Claude need to plan well?)
    'why_git_log_injection': '...',

    # When would you use /plan vs just asking "how should I implement X?"
    # Hint: think about what gets saved/documented
    'plan_vs_direct_ask': '...',

    # Why does /plan NOT have tool restrictions (like skills can)?
    # What does this mean Claude could do if the stop gate weren't there?
    'no_tool_restriction_risk': '...',
}

for q, a in plan_analysis.items():
    print(f'{q}:\n  {a}\n')

<details>
<summary>✅ <b>Solution</b> (click to expand)</summary>

```python
plan_analysis = {
    'why_stop_gate': (
        "Without it, Claude treats plan + implementation as a single task and "
        "starts writing code immediately after the plan. The stop gate forces a "
        "human review checkpoint between thinking and acting."
    ),
    'why_git_log_injection': (
        "git log shows what work is already done (avoids duplicating) and what "
        "branch/state the repo is in. git diff --stat shows what's in flight — "
        "useful if the user is mid-feature."
    ),
    'plan_vs_direct_ask': (
        "Use /plan when you want the plan persisted in the conversation as a "
        "confirmed agreement — it becomes a shared reference. Direct asking is "
        "more exploratory and less committed."
    ),
    'no_tool_restriction_risk': (
        "Without tool restrictions, Claude could write files during the planning "
        "phase. The stop gate is the only safeguard — which is why the wording "
        "must be explicit and firm."
    ),
}
```

</details>

### Exercise 5.5.3 — Compare Commands vs Skills

The repo has both `.claude/commands/` and `.claude/skills/`. Read one command and one skill with similar purposes and explain the tradeoff.

In [ ]:
# Read /tdd (command) and /debug (skill) — both are systematic debugging/development workflows

print('=== /tdd COMMAND ===')
print((CLAUDE_DIR / 'commands' / 'tdd.md').read_text())
print()
print('=== /debug SKILL ===')
print((CLAUDE_DIR / 'skills' / 'debug' / 'SKILL.md').read_text())

In [ ]:
# TODO: Fill in the comparison table

comparison = {
    '/tdd (command)': {
        'format': '...',         # flat .md / SKILL.md with frontmatter
        'tool_restrictions': '...', # yes/no — which tools?
        'model_override': '...',  # yes/no
        'context_isolation': '...', # yes/no (context: fork)
        'dynamic_injection': '...', # yes/no — what does it inject?
        'best_for': '...',
    },
    '/debug (skill)': {
        'format': '...',
        'tool_restrictions': '...',
        'model_override': '...',
        'context_isolation': '...',
        'dynamic_injection': '...',
        'best_for': '...',
    },
}

for name, props in comparison.items():
    print(f'\n{name}:')
    for k, v in props.items():
        print(f'  {k}: {v}')

<details>
<summary>✅ <b>Solution</b> (click to expand)</summary>

```python
comparison = {
    '/tdd (command)': {
        'format': 'flat .md — no frontmatter',
        'tool_restrictions': 'none — inherits all tools',
        'model_override': 'no',
        'context_isolation': 'no — runs in main context',
        'dynamic_injection': 'yes — !`uv run pytest --collect-only -q`',
        'best_for': 'structured multi-phase workflow with live test status injected',
    },
    '/debug (skill)': {
        'format': 'SKILL.md with YAML frontmatter',
        'tool_restrictions': 'allowed-tools: Read, Bash, Grep, Glob (no Write)',
        'model_override': 'no (inherits)',
        'context_isolation': 'context: fork — isolated subagent',
        'dynamic_injection': 'no',
        'best_for': 'investigation that should not pollute main context with exploration tokens',
    },
}
```

Key insight: `/tdd` injects live test state so Claude knows exactly where it stands. `/debug` isolates exploration into a fork so verbose grep/read operations don't crowd your conversation.

</details>

### Exercise 5.5.4 — Write an `/eval-ai` Command

The `/eval` command in this repo is a general-purpose eval framework. Write a specialized command `/eval-ai` tailored specifically to AI features in this curriculum.

It should define eval criteria specific to AI systems: latency, token cost, accuracy, and regression on known test cases.

In [ ]:
# TODO: Create .claude/commands/eval-ai.md
#
# Requirements:
#   - Dynamic injection: show current evals with !`ls .claude/evals/ 2>/dev/null`
#   - Operations: define / check / report  (same as /eval but AI-specific)
#   - For `define`: include AI-specific criteria sections:
#       * Accuracy (pass@1 on labeled test set)
#       * Latency (p50, p95 in ms)
#       * Token cost (avg tokens per call)
#       * Regression (comparison to baseline model responses)
#   - For `check`: run the actual Python evaluation script if it exists
#   - For `report`: include a "SHIP / DO NOT SHIP" recommendation with thresholds:
#       * pass@1 >= 80%, latency p95 < 2000ms, cost regression < 10%
#   - End with $ARGUMENTS substitution for the feature name

eval_ai_content = """# /eval-ai — AI Feature Evaluation

# TODO: Write the command here
"""

(CLAUDE_DIR / 'commands' / 'eval-ai.md').write_text(eval_ai_content)
print('Created .claude/commands/eval-ai.md')
print()
print((CLAUDE_DIR / 'commands' / 'eval-ai.md').read_text())

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

Structure it like `/eval` but with AI-specific sections. For the `define` operation, include a template that has fields for accuracy threshold, latency budget, and cost ceiling. For `check`, use a shell injection to show existing eval logs.

The key addition over generic `/eval` is the **latency** and **token cost** columns — AI systems fail in production due to cost creep and slow p95 latency, not just accuracy.

</details>

<details>
<summary>✅ <b>Solution</b> (click to expand)</summary>

````markdown
# /eval-ai — AI Feature Evaluation

Evaluate AI-powered features against measurable quality gates.

## Existing evals
```
!`ls .claude/evals/ 2>/dev/null || echo "No evals yet — run /eval-ai define <feature>"`
```

## Operations: `$ARGUMENTS`

---

### `define <feature-name>`

Create `.claude/evals/<feature-name>.md`:

```markdown
# Eval: <feature-name>

## Accuracy (correctness)
- [ ] Test case 1: [input] → [expected output]
- Threshold: pass@1 >= 80%

## Latency
- p50 target: < 500ms
- p95 target: < 2000ms

## Token cost
- Max avg tokens/call: [N]
- Cost regression limit: +10% vs baseline

## Regression tests
- [ ] [Existing behavior that must not break]

## Baseline (fill after first run)
| Metric | Baseline | Current | Delta |
|--------|---------|---------|-------|
| pass@1 | — | — | — |
| p95 latency | — | — | — |
| avg tokens | — | — | — |
```

---

### `check <feature-name>`

Run evaluation against `.claude/evals/<feature-name>.md`.
For each test case, call the feature, compare output, record latency and token count.
Report: `pass@1: X/N | p95: Xms | avg tokens: N | vs baseline: ±X%`

---

### `report`

Read all `.claude/evals/*.md`. Produce a table:

| Feature | pass@1 | p95 | tokens | Status |
|---------|--------|-----|--------|--------|

**SHIP** if: all pass@1 ≥ 80%, all p95 < 2000ms, no cost regression > 10%
**DO NOT SHIP** otherwise. List blocking issues.
````

</details>

---
## Part 6: Full Configuration Audit

In [ ]:
# Complete audit of the .claude/ directory

def audit_claude_config(claude_dir: Path, repo_root: Path) -> dict:
    report = {'sections': {}}

    # CLAUDE.md
    claude_md = repo_root / 'CLAUDE.md'
    content = claude_md.read_text()
    imports = re.findall(r'^@(.+)$', content, re.MULTILINE)
    report['sections']['claude_md'] = {
        'lines': len(content.splitlines()),
        'imports': imports,
        'ok': len(content.splitlines()) <= 200
    }

    # Contexts
    ctx_dir = claude_dir / 'contexts'
    contexts = []
    for f in sorted(ctx_dir.glob('*.md')):
        contexts.append({'name': f.name, 'lines': len(f.read_text().splitlines())})
    report['sections']['contexts'] = contexts

    # Rules
    rules = []
    for f in sorted((claude_dir / 'rules').glob('*.md')):
        c = f.read_text()
        scoped = 'paths:' in c[:200]
        rules.append({'name': f.name, 'scoped': scoped, 'lines': len(c.splitlines())})
    report['sections']['rules'] = rules

    # Hooks
    hooks = []
    for f in sorted((claude_dir / 'hooks').glob('*.sh')):
        hooks.append({'name': f.name, 'executable': os.access(f, os.X_OK), 'lines': len(f.read_text().splitlines())})
    report['sections']['hooks'] = hooks

    # Settings wiring
    settings_file = claude_dir / 'settings.json'
    if settings_file.exists():
        s = json.loads(settings_file.read_text())
        all_hook_cmds = []
        for groups in s.get('hooks', {}).values():
            for group in groups:
                for h in group.get('hooks', []):
                    if 'command' in h:
                        all_hook_cmds.append(Path(h['command'].split('/')[-1]))
        wired = {cmd.name for cmd in all_hook_cmds}
        unwired = {f.name for f in (claude_dir / 'hooks').glob('*.sh')} - wired
        report['sections']['settings'] = {'wired_scripts': list(wired), 'unwired': list(unwired)}

    # Skills
    skills = []
    for d in sorted((claude_dir / 'skills').iterdir()):
        sf = d / 'SKILL.md'
        if sf.exists():
            p = parse_skill(sf)
            skills.append({'name': p['frontmatter'].get('name', d.name), 'manual': p['frontmatter'].get('disable-model-invocation') == 'true', 'lines': p['body_lines']})
    report['sections']['skills'] = skills


    # Commands
    commands_dir = claude_dir / 'commands'
    commands = []
    if commands_dir.exists():
        for f in sorted(commands_dir.glob('*.md')):
            content = f.read_text()
            injections = sum(1 for l in content.splitlines() if l.strip().startswith('!`'))
            has_stop = 'STOP' in content.upper() and 'WAIT' in content.upper()
            commands.append({'name': f.stem, 'lines': len(content.splitlines()),
                             'injections': injections, 'has_stop': has_stop})
    report['sections']['commands'] = commands

    # Agents
    agents = []
    for f in sorted((claude_dir / 'agents').glob('*.md')):
        p = parse_agent(f)
        agents.append({'name': f.stem, 'model': p['fm'].get('model', 'inherit'), 'memory': p['fm'].get('memory', '-'), 'lines': p['body_lines']})
    report['sections']['agents'] = agents

    # MCP
    mcp_file = repo_root / '.mcp.json'
    if mcp_file.exists():
        mcp = json.loads(mcp_file.read_text())
        report['sections']['mcp'] = list(mcp.get('mcpServers', {}).keys())

    return report

report = audit_claude_config(CLAUDE_DIR, REPO_ROOT)

print('╔══════════════════════════════════════════════╗')
print('║     .claude/ Configuration Audit             ║')
print('╚══════════════════════════════════════════════╝')

cm = report['sections']['claude_md']
cm_status = '✓' if cm['ok'] else '⚠ OVER 200 LINES'
print(f'\nCLAUDE.md: {cm["lines"]} lines {cm_status}')
print(f'  @imports: {len(cm["imports"])} → {cm["imports"]}')

print(f'\nContexts ({len(report["sections"]["contexts"])}):')
for c in report['sections']['contexts']:
    print(f'  {c["name"]:35s} {c["lines"]} lines')

print(f'\nRules ({len(report["sections"]["rules"])}):')
for r in report['sections']['rules']:
    scoped = '(path-scoped)' if r['scoped'] else '(always loaded)'
    print(f'  {r["name"]:35s} {scoped}')

print(f'\nHooks ({len(report["sections"]["hooks"])}):')
for h in report['sections']['hooks']:
    status = '✓' if h['executable'] else '✗ NOT EXECUTABLE'
    print(f'  {status}  {h["name"]:35s} {h["lines"]} lines')

s = report['sections'].get('settings', {})
unwired = s.get('unwired', [])
print(f'\nsettings.json wiring:')
print(f'  Wired scripts: {len(s.get("wired_scripts", []))}')
if unwired:
    print(f'  WARNING: Unwired scripts: {unwired}')
else:
    print('  All hooks are wired ✓')

print(f'\nSkills ({len(report["sections"]["skills"])}):')
for sk in report['sections']['skills']:
    inv = 'manual-only' if sk['manual'] else 'auto+manual'
    print(f'  /{sk["name"]:20s} {inv}')

print(f'\nAgents ({len(report["sections"]["agents"])}):')
for ag in report['sections']['agents']:
    print(f'  {ag["name"]:25s} model:{ag["model"]:9} memory:{ag["memory"]}')


print(f'\nCommands ({len(report["sections"]["commands"])}):')
for cmd in report['sections']['commands']:
    stop = '(has stop gate)' if cmd['has_stop'] else ''
    inj = f'{cmd["injections"]} injection(s)' if cmd['injections'] else 'no injection'
    print(f'  /{cmd["name"]:25s} {inj:20s} {stop}')

print(f'\nMCP servers: {report["sections"].get("mcp", [])}')

# Summary
total = (len(report['sections']['contexts']) + len(report['sections']['rules']) +
         len(report['sections']['hooks']) + len(report['sections']['skills']) +
         len(report['sections']['commands']) + len(report['sections']['agents']))
print(f'\n═════ Total: {total} configured components ═════')

### Exercise 6.1 — Reflection Questions

**Q1:** The `format-python.sh` hook runs `async: true`. What would happen if it ran synchronously (blocking)? When would you choose blocking vs. async for a hook?

*Your answer here...*

<details>
<summary>💡 <b>Model Answer</b> (click to expand)</summary>

Synchronously, `format-python.sh` would pause Claude after every file edit until ruff finishes — adding ~100-300ms to each write. Across a session with 50 file edits, that's 5-15 seconds of dead time.

**Use blocking (sync) when:** the hook's result affects Claude's next action — e.g., `detect-secrets.sh` must be blocking because Claude should not proceed if a secret is found.

**Use async when:** the hook is informational or side-effecting but Claude doesn't need to wait — formatting, logging, test runs. These fire and forget.

</details>

**Q2:** `doc-researcher` uses `model: haiku` and `code-reviewer` uses `model: sonnet`. The `doc-researcher` actually reads more files. Why does the simpler task get the cheaper model?

*Your answer here...*

<details>
<summary>💡 <b>Model Answer</b> (click to expand)</summary>

Model choice is about **reasoning depth**, not file count. `doc-researcher` is doing retrieval and extraction — finding where topics are covered, reading headers, matching terms. This is pattern-matching work that Haiku handles well.

`code-reviewer` must understand subtle bugs: off-by-one errors in embedding dimension handling, prompt injection in string concatenation, N+1 patterns in async code. This requires Sonnet's stronger reasoning even if it reads fewer files.

Rule: match model to **reasoning complexity**, not **I/O volume**.

</details>

**Q3:** The contexts in `.claude/contexts/` contain the same information as CLAUDE.md would if you wrote it all inline. What's the *real* reason to split them out — beyond the 200-line limit?

*Your answer here...*

<details>
<summary>💡 <b>Model Answer</b> (click to expand)</summary>

**Maintainability and ownership.** When model names change (every few months in this space), only `models.md` needs updating — not CLAUDE.md. When you add a new week, only `architecture.md` changes. Different team members can own different context files.

Also: **composability**. A different CLAUDE.md (e.g., for a student's fork) can `@import` only the contexts it needs. A team adding a backend service can import `architecture.md` and `security.md` but not `lab-conventions.md`.

The 200-line limit is the constraint that forces this — but the modularity is the actual benefit.

</details>

---
## Summary

You explored and extended a complete Claude Code configuration:

| Pillar | Files | What you did |
|--------|-------|-------------|
| **Memory** | `CLAUDE.md`, `.claude/contexts/`, `.claude/rules/` | Mapped the session load, extended `models.md`, analyzed path scoping |
| **Hooks** | `.claude/hooks/` × 6, `settings.json` | Tested `detect-secrets.sh` live, extended it to catch uppercase patterns |
| **Skills** | `.claude/skills/` × 6 | Inventoried all commands, analyzed `/review` design, wrote `/weekly-summary` |
| **Agents** | `.claude/agents/` × 4 | Compared allowlist vs. denylist, wrote `dependency-auditor` |
| **MCP** | `.mcp.json` | Read 5 server configs, matched them to curriculum use cases |
| **Commands** | `.claude/commands/` × 5 | Inventoried workflow macros, analyzed the `/plan` stop gate, compared commands vs skills, wrote `/eval-ai` |

**The key insight:** configuration is code. A hook that catches one class of bug on every edit, a command that enforces planning discipline, a skill that packages your most-repeated workflow, an agent that accumulates codebase knowledge across sessions — these compound over time in ways that raw prompting cannot.

**Try these now in Claude Code:**
- `/plan "add rate limiting to the OpenAI client"` — see the stop gate in action
- `/tdd "implement embedding cache"` — enforce red→green→refactor
- `/eval define semantic-search` — start measuring your AI features
- `/lab-status` — see your progress across all labs
- `@"code-reviewer (agent)" review the changes from this lab`

**Next:** Week 4 — Structured Output with Pydantic